# 3장 — 사전학습, 생성, SFT, GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch03_pretrain_sft_grpo.ipynb)

이 노트북은 『밑바닥부터 시작하는 딥러닝 6』의 공식 코드 저장소를 기준으로 구성했습니다. T4에서 실행하기 어렵다는 이유로 알고리즘이나 모델 구조를 토이 버전으로 바꾸지 않습니다.

- 기준 upstream commit: `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`
- 포함한 장 코드 파일 수: **6개**
- 함께 펼쳐서 보여주는 공통 모듈 수: **3개**


## 노트북 구성 원칙

1. 공식 `.py`의 모델 구조와 계산 로직을 그대로 유지합니다.
2. 함수·클래스·실행부를 셀 단위로 나눠 위에서 아래로 읽기 쉽게 배치합니다.
3. 일본어 자연어 주석은 한국어로 바꾸며, 변수명·수식·텐서 shape 같은 기술 표기는 유지합니다.
4. 공통 `codebot` / `storybot` 모듈도 외부 파일 뒤에 숨기지 않고 이 노트북에서 직접 확인할 수 있게 합니다.
5. T4에서 시간이 오래 걸리는 전체 학습 스케줄도 기본값 자체를 임의 축소하지 않습니다.


## 0. Colab 환경 준비

먼저 공식 저장소를 고정된 커밋으로 준비하고 현재 런타임의 GPU를 확인합니다.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('작업 경로:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 확인 중 오류:', exc)


## 1. 이 장에서 사용하는 공통 구현

장 코드가 import하는 로컬 모듈을 먼저 읽습니다. 긴 파일도 클래스·함수 단위로 나눠 표시합니다.


### `codebot/model.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile codebot/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


#### `MultiHeadAttention` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, dropout_rate=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)  # 출력 예시: (B, C, H*D)
        K = self.W_k(x)  # 출력 예시: (B, C, H*D)
        V = self.W_v(x)  # 출력 예시: (B, C, H*D)

        Q = Q.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)
        K = K.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)
        V = V.view(B, C, H, D).transpose(1, 2)  # 출력 예시: (B, H, C, D)

        scores = torch.matmul(Q, K.transpose(-2, -1))  # 출력 예시: (B, H, C, C)
        scores = scores / (D ** 0.5)

        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)  # 출력 예시: (B, H, C, C)
        weights = self.attention_dropout(weights)
        hidden = torch.matmul(weights, V)  # 출력 예시: (B, H, C, D)

        hidden = hidden.transpose(1, 2).contiguous()  # 출력 예시: (B, C, H, D)
        hidden = hidden.view(B, C, H * D)  # 출력 예시: (B, C, H*D)
        output = self.W_o(hidden)  # 출력 예시: (B, C, E)
        output = self.output_dropout(output)

        return output


#### `LayerNorm` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta


#### `GELU` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


#### `FFN` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class FFN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),  # 참고: GELU()
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.layers(x)


#### `Block` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.LayerNorm(embed_dim)  # 참고: LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = nn.LayerNorm(embed_dim)  # 참고: LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


#### `GPT` 클래스 구현


In [ ]:
%%writefile -a codebot/model.py


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, dropout_rate):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_context_len, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, dropout_rate)
            for _ in range(n_layer)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size)

        self.embed.weight = self.unembed.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, C = ids.shape
        device = ids.device

        pos = torch.arange(0, C, dtype=torch.long, device=device)
        emb = self.embed(ids)
        pos_emb = self.pos_embed(pos)
        x = self.dropout(emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        logits = self.unembed(x)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            dropout_rate=checkpoint['dropout_rate']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model


### `codebot/tokenizer.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile codebot/tokenizer.py
import regex as re
from collections import defaultdict
import pickle
from tqdm import tqdm


#### `pretokenize()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)


#### `count_pairs()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py

def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts


#### `merge()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


#### `train_bpe()` 함수 구현


In [ ]:
%%writefile -a codebot/tokenizer.py

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    texts = input_text.split(end_token)

    ids_list = []
    for text in texts:
        for pretoken in pretokenize(text):
            ids_list.append(list(pretoken.encode("utf-8")))

    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        if not counts:
            break

        # 참고: best_pair = max(counts, key=counts.get)
        best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules


#### `BPETokenizer` 클래스 구현


In [ ]:
%%writefile -a codebot/tokenizer.py


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # 이 코드 단계의 동작을 확인하는 예시
        texts = tqdm(texts, desc="Encoding") if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### `codebot/utils.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile codebot/utils.py
import torch
import torch.nn.functional as F


#### `generate()` 함수 구현


In [ ]:
%%writefile -a codebot/utils.py


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()  # 이 코드 단계의 동작을 확인하는 예시

    # 이 코드 단계의 동작을 확인하는 예시
    device = next(model.parameters()).device  # 이 코드 단계의 동작을 확인하는 예시
    ids = tokenizer.encode(prompt)
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    # 이 코드 단계의 동작을 확인하는 예시
    generated_ids = ids.clone()

    # 이 코드 단계의 동작을 확인하는 예시
    for _ in range(max_new_tokens):
        # 이 코드 단계의 동작을 확인하는 예시
        if ids.size(1) > model.max_context_len:
            ids = ids[:, -model.max_context_len:]

        # 이 코드 단계의 동작을 확인하는 예시
        logits = model(ids)[:, -1, :]
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        # 이 코드 단계의 동작을 확인하는 예시
        if next_id.item() == tokenizer.end_token_id:
            break

        # 이 코드 단계의 동작을 확인하는 예시
        ids = torch.cat((ids, next_id), dim=1)
        generated_ids = torch.cat((generated_ids, next_id), dim=1)

    # 이 코드 단계의 동작을 확인하는 예시
    generated_text = tokenizer.decode(generated_ids[0].tolist())
    return generated_text


#### `get_device()` 함수 구현


In [ ]:
%%writefile -a codebot/utils.py

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')


## 2. 장별 실습 코드

공식 저장소의 장 코드를 파일 순서대로 모두 다룹니다.


## `ch03/01_pretrain.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from itertools import cycle
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from codebot.model import GPT
from codebot.utils import get_device


### 설정 및 값 준비: `device`


In [ ]:

# 설정
device = get_device()


### 설정 및 값 준비: `data_path`


In [ ]:
data_path = 'codebot/tiny_codes.bin'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'codebot/merge_rules.pkl'


### 설정 및 값 준비: `model_save_path`


In [ ]:
model_save_path = 'codebot/model_pretrain.pt'


### 설정 및 값 준비: `context_len`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
context_len = 256


### 설정 및 값 준비: `vocab_size`


In [ ]:
vocab_size = 1000


### 설정 및 값 준비: `batch_size`


In [ ]:
batch_size = 32


### 설정 및 값 준비: `learning_rate`


In [ ]:
learning_rate = 3e-4


### 설정 및 값 준비: `max_iters`


In [ ]:
max_iters = 20000


### 설정 및 값 준비: `embed_dim`


In [ ]:
embed_dim = 384


### 설정 및 값 준비: `n_head`


In [ ]:
n_head = 6


### 설정 및 값 준비: `n_layer`


In [ ]:
n_layer = 6


### 설정 및 값 준비: `ff_dim`


In [ ]:
ff_dim = 4 * embed_dim


### 설정 및 값 준비: `dropout_rate`


In [ ]:
dropout_rate = 0.1


### `TokenDataset` 클래스 구현


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
class TokenDataset(Dataset):
    def __init__(self, tokens, context_len):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.context_len = context_len

    def __len__(self):
        return len(self.tokens) - self.context_len

    def __getitem__(self, idx):
        x = self.tokens[idx:idx+self.context_len]
        y = self.tokens[idx+1:idx+self.context_len+1]
        return x, y


### 설정 및 값 준비: `ids`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
ids = np.fromfile(data_path, dtype=np.uint16)


### 설정 및 값 준비: `dataset`


In [ ]:
dataset = TokenDataset(ids, context_len)


### 설정 및 값 준비: `dataloader`


In [ ]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


### 설정 및 값 준비: `model`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model = GPT(
    vocab_size=vocab_size,
    max_context_len=context_len,
    embed_dim=embed_dim,
    n_head=n_head,
    n_layer=n_layer,
    ff_dim=ff_dim,
    dropout_rate=dropout_rate
).to(device)


### 설정 및 값 준비: `optimizer`


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


### 설정 및 값 준비: `total_params`


In [ ]:

total_params = sum(p.numel() for p in model.parameters())


### 실행 및 결과 확인


In [ ]:
print(f"パラメータ数: {total_params:,} ({total_params/1e6:.1f}M)")


### 설정 및 값 준비: `losses`


In [ ]:

losses = []


### 설정 및 값 준비: `data_iter`


In [ ]:
data_iter = cycle(dataloader)  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `pbar`


In [ ]:
pbar = tqdm(range(max_iters))


### 반복 실행


In [ ]:

for i in pbar:
    batch_x, batch_y = next(data_iter)
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    logits = model(batch_x)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
plt.figure(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:
plt.plot(losses)


### 실행 및 결과 확인


In [ ]:
plt.xlabel('Iteration')


### 실행 및 결과 확인


In [ ]:
plt.ylabel('Loss')


### 실행 및 결과 확인


In [ ]:
plt.grid(True)


### 실행 및 결과 확인


In [ ]:
plt.savefig('loss_pretrain.png')


### 실행 및 결과 확인


In [ ]:

model.save(model_save_path)


## `ch03/02_generate.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os
import sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import torch
import torch.nn.functional as F
from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import get_device


### 설정 및 값 준비: `device`


In [ ]:


# 설정
device = get_device()


### 설정 및 값 준비: `model_path`


In [ ]:
model_path = 'codebot/model_pretrain.pt'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'codebot/merge_rules.pkl'


### 설정 및 값 준비: `prompt`


In [ ]:

# 생성설정
prompt = "def"  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `max_new_tokens`


In [ ]:
max_new_tokens = 200  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `temperature`


In [ ]:
temperature = 1.0  # 이 코드 단계의 동작을 확인하는 예시


### `generate()` 함수 구현


In [ ]:

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()  # 이 코드 단계의 동작을 확인하는 예시

    # 이 코드 단계의 동작을 확인하는 예시
    device = next(model.parameters()).device  # 이 코드 단계의 동작을 확인하는 예시
    ids = tokenizer.encode(prompt)
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    # 이 코드 단계의 동작을 확인하는 예시
    generated_ids = ids.clone()

    # 이 코드 단계의 동작을 확인하는 예시
    for _ in range(max_new_tokens):
        # 이 코드 단계의 동작을 확인하는 예시
        if ids.size(1) > model.max_context_len:
            ids = ids[:, -model.max_context_len:]

        # 이 코드 단계의 동작을 확인하는 예시
        logits = model(ids)[:, -1, :]
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        # 이 코드 단계의 동작을 확인하는 예시
        if next_id.item() == tokenizer.end_token_id:
            break

        # 이 코드 단계의 동작을 확인하는 예시
        ids = torch.cat((ids, next_id), dim=1)
        generated_ids = torch.cat((generated_ids, next_id), dim=1)

    # 이 코드 단계의 동작을 확인하는 예시
    generated_text = tokenizer.decode(generated_ids[0].tolist())
    return generated_text


### 설정 및 값 준비: `tokenizer`


In [ ]:

tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model`


In [ ]:
model = GPT.load_from(model_path, device=device)


### 반복 실행


In [ ]:

# 텍스트생성
for i in range(5):
    print(f"--- サンプル {i+1} ---")
    generated_text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature
    )
    print(generated_text)
    print()


## `ch03/03_alpaca.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import json
from codebot.tokenizer import BPETokenizer


### 설정 및 값 준비: `tokenizer`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer.load_from('codebot/merge_rules.pkl')


### 실행 코드


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
with open('codebot/tiny_codes_sft.json') as f:
    data = json.load(f)


### 설정 및 값 준비: `item`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
item = data[0]


### 실행 및 결과 확인


In [ ]:
print(item)


### 설정 및 값 준비: `text`


In [ ]:
# 출력 예시: {'instruction': 'Hello', 'response': 'Hello. What can I help you with?'}

# 이 코드 단계의 동작을 확인하는 예시
text = f"### Instruction:\n{item['instruction']}\n\n### Response:\n{item['response']}<|endoftext|>"


### 실행 및 결과 확인


In [ ]:
print(text)


### 설정 및 값 준비: `ids`


In [ ]:
# ### Instruction:
# 텐서 크기: Hello
#
# ### Response:
# Hello. What can I help you with?<|endoftext|>

# 이 코드 단계의 동작을 확인하는 예시
ids = tokenizer.encode(text)


### 실행 및 결과 확인


In [ ]:
print(ids)


### 마지막 실행 코드


In [ ]:
# 출력 예시: [35, 35, 35, 962, 519, 117, 389, 58, 10, 846, 10, 10, 35, 35, 35, 752, 568, 58, 10, 846, 46, 840, 104, 277, 280, 356, 473, 708, 108, 112, 930, 657, 63, 999]


## `ch03/03_sft.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from itertools import cycle
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import get_device


### 설정 및 값 준비: `device`


In [ ]:

# 설정
device = get_device()


### 설정 및 값 준비: `data_path`


In [ ]:
data_path = 'codebot/tiny_codes_sft.json'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'codebot/merge_rules.pkl'


### 설정 및 값 준비: `pretrain_model_path`


In [ ]:
pretrain_model_path = 'codebot/model_pretrain.pt'


### 설정 및 값 준비: `sft_model_save_path`


In [ ]:
sft_model_save_path = 'codebot/model_sft.pt'


### 설정 및 값 준비: `context_len`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
context_len = 256


### 설정 및 값 준비: `batch_size`


In [ ]:
batch_size = 32


### 설정 및 값 준비: `learning_rate`


In [ ]:
learning_rate = 3e-4


### 설정 및 값 준비: `max_iters`


In [ ]:
max_iters = 500


### `SFTDataset` 클래스 구현


In [ ]:

class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, context_len):
        self.tokenizer = tokenizer
        self.context_len = context_len
        self.samples = []

        with open(data_path) as f:
            data = json.load(f)

        for item in data:
            ids, labels = self._create_sample(item['instruction'], item['response'])
            self.samples.append((ids, labels))

    def _create_sample(self, instruction, response):
        # 이 코드 단계의 동작을 확인하는 예시
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
        response = f"{response}<|endoftext|>"

        # 이 코드 단계의 동작을 확인하는 예시
        prompt_ids = self.tokenizer.encode(prompt)
        response_ids = self.tokenizer.encode(response)

        # 이 코드 단계의 동작을 확인하는 예시
        ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids

        # 이 코드 단계의 동작을 확인하는 예시
        ids = ids[:-1]
        labels = labels[1:]

        # 이 코드 단계의 동작을 확인하는 예시
        pad_len = self.context_len - len(ids)
        if pad_len > 0:
            ids = ids + [0] * pad_len  # 이 코드 단계의 동작을 확인하는 예시
            labels = labels + [-100] * pad_len
        elif pad_len < 0:
            ids = ids[:self.context_len]
            labels = labels[:self.context_len]

        return ids, labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, labels = self.samples[idx]
        return torch.tensor(ids, dtype=torch.long),\
               torch.tensor(labels, dtype=torch.long)


### 설정 및 값 준비: `tokenizer`


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `dataset`


In [ ]:
dataset = SFTDataset(data_path, tokenizer, context_len)


### 설정 및 값 준비: `dataloader`


In [ ]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


### 설정 및 값 준비: `model`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model = GPT.load_from(pretrain_model_path, device=device)


### 설정 및 값 준비: `optimizer`


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


### 설정 및 값 준비: `losses`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
losses = []


### 설정 및 값 준비: `data_iter`


In [ ]:
data_iter = cycle(dataloader)


### 설정 및 값 준비: `pbar`


In [ ]:
pbar = tqdm(range(max_iters))


### 반복 실행


In [ ]:

for i in pbar:
    batch_x, batch_y = next(data_iter)
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    logits = model(batch_x)
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        batch_y.view(-1),
        ignore_index=-100
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
plt.figure(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:
plt.plot(losses)


### 실행 및 결과 확인


In [ ]:
plt.xlabel('Iteration')


### 실행 및 결과 확인


In [ ]:
plt.ylabel('Loss')


### 실행 및 결과 확인


In [ ]:
plt.grid(True)


### 실행 및 결과 확인


In [ ]:
plt.savefig('loss_sft.png')


### 실행 및 결과 확인


In [ ]:

# 모델의저장
model.save(sft_model_save_path)


## `ch03/04_chat.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import generate, get_device


### 설정 및 값 준비: `device`


In [ ]:

# 설정
device = get_device()


### 설정 및 값 준비: `model_path`


In [ ]:
model_path = 'codebot/model_sft.pt'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
# model_path = 'codebot/model_grpo.pt'
tokenizer_path = 'codebot/merge_rules.pkl'


### 설정 및 값 준비: `max_new_tokens`


In [ ]:
max_new_tokens = 200


### 설정 및 값 준비: `temperature`


In [ ]:
temperature = 1.0


### `format_prompt()` 함수 구현


In [ ]:

def format_prompt(user_message):
    return f"### Instruction:\n{user_message}\n\n### Response:\n"


### 설정 및 값 준비: `tokenizer`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model`


In [ ]:
model = GPT.load_from(model_path, device=device)


### 반복 실행


In [ ]:

while True:
    user_input = input("\nYou: ").strip()

    if not user_input:
        continue

    # 이 코드 단계의 동작을 확인하는 예시
    prompt = format_prompt(user_input)
    response = generate(model, tokenizer, prompt, max_new_tokens, temperature)

    # 이 코드 단계의 동작을 확인하는 예시
    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()

    # 이 코드 단계의 동작을 확인하는 예시
    if "\n" in response:
        print(f"Bot:\n{response}")
    else:
        print(f"Bot: {response}")


## `ch03/09_grpo.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from itertools import cycle
import re
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import generate, get_device


### `GRPODataset` 클래스 구현


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
class GRPODataset(Dataset):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.data = []
        for i in range(1, 10):
            for j in range(1, 10):
                prompt = f"### Instruction:\n{i}+{j}=\n\n### Response:\n"
                ground_truth = i + j
                self.data.append((prompt, ground_truth))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

    def get_batch(self, prompts, responses, device):
        all_ids = []
        all_masks = []

        for prompt, response in zip(prompts, responses):
            prompt_ids = self.tokenizer.encode(prompt)
            response_ids = self.tokenizer.encode(response)

            ids = prompt_ids + response_ids
            mask = [0] * len(prompt_ids) + [1] * len(response_ids)

            all_ids.append(ids)
            all_masks.append(mask)

        # 이 코드 단계의 동작을 확인하는 예시
        max_len = max(len(ids) for ids in all_ids)
        padded_ids = []
        padded_masks = []
        for ids, mask in zip(all_ids, all_masks):
            pad_len = max_len - len(ids)
            padded_ids.append(ids + [0] * pad_len)
            padded_masks.append(mask + [0] * pad_len)

        ids = torch.tensor(padded_ids, dtype=torch.long, device=device)
        mask = torch.tensor(padded_masks, dtype=torch.float, device=device)

        return ids, mask


### `calculate_reward()` 함수 구현


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
def calculate_reward(ground_truth, response):
    try:
        matches = re.findall(r'(-?\d+)', response)
        if matches:
            predicted = int(matches[-1])  # 이 코드 단계의 동작을 확인하는 예시
            return 1.0 if predicted == ground_truth else 0.0
        return 0.0
    except:
        return 0.0


### `generate_group()` 함수 구현


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
def generate_group(model, tokenizer, prompts, gts, group_size):
    all_prompts = []
    all_responses = []
    all_advantages = []

    for prompt, gt in zip(prompts, gts):
        responses = []
        for _ in range(group_size):
            full_text = generate(model, tokenizer, prompt, temperature=1.0)
            response = full_text[len(prompt):]
            responses.append(response)

        rewards = torch.tensor([calculate_reward(gt, r) for r in responses])
        advantages = rewards - rewards.mean()

        for response, advantage in zip(responses, advantages):
            all_prompts.append(prompt)
            all_responses.append(response)
            all_advantages.append(advantage)

    return all_prompts, all_responses, torch.stack(all_advantages)


### `compute_probs()` 함수 구현


In [ ]:

# 손실함수
def compute_probs(model, ids):
    logits = model(ids)  # 출력 예시: (B, C, V)
    probs = F.softmax(logits[:, :-1, :], dim=-1)  # 출력 예시: (B, C-1, V)
    labels = ids[:, 1:]  # 출력 예시: (B, C-1)

    token_probs = torch.gather(
        probs, dim=-1, index=labels.unsqueeze(-1)
    ).squeeze(-1)  # 출력 예시: (B, C-1)

    return token_probs


### `grpo_loss()` 함수 구현


In [ ]:

def grpo_loss(model, old_model, ids, mask, advantages, epsilon=0.2):
    # 이 코드 단계의 동작을 확인하는 예시
    probs = compute_probs(model, ids)
    # 이 코드 단계의 동작을 확인하는 예시
    with torch.no_grad():
        old_probs = compute_probs(old_model, ids)

    # 이 코드 단계의 동작을 확인하는 예시
    ratio = probs / (old_probs + 1e-8)
    advantages = advantages.unsqueeze(-1)

    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages

    mask = mask[:, 1:]  # 이 코드 단계의 동작을 확인하는 예시
    token_objective = torch.min(unclipped, clipped) * mask

    # 이 코드 단계의 동작을 확인하는 예시
    n_samples = ids.size(0)  # batch_size × group_size
    return -token_objective.sum() / n_samples


### 설정 및 값 준비: `device`


In [ ]:


# 설정
device = get_device()


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'codebot/merge_rules.pkl'


### 설정 및 값 준비: `sft_model_path`


In [ ]:
sft_model_path = 'codebot/model_sft.pt'


### 설정 및 값 준비: `grpo_model_save_path`


In [ ]:
grpo_model_save_path = 'codebot/model_grpo.pt'


### 설정 및 값 준비: `learning_rate`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
learning_rate = 7e-6


### 설정 및 값 준비: `max_iters`


In [ ]:
max_iters = 500


### 설정 및 값 준비: `n_update_per_generation`


In [ ]:
n_update_per_generation = 2  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `eval_interval`


In [ ]:
eval_interval = 10


### 설정 및 값 준비: `epsilon`


In [ ]:
epsilon = 0.2  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `group_size`


In [ ]:
group_size = 8  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `batch_size`


In [ ]:
batch_size = 32


### 설정 및 값 준비: `tokenizer`


In [ ]:

# 초기화
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model`


In [ ]:
model = GPT.load_from(sft_model_path, device=device)


### 설정 및 값 준비: `optimizer`


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


### 설정 및 값 준비: `old_model`


In [ ]:

old_model = GPT.load_from(sft_model_path, device=device)  # 이 코드 단계의 동작을 확인하는 예시


### 실행 및 결과 확인


In [ ]:
old_model.eval()


### 설정 및 값 준비: `dataset`


In [ ]:

dataset = GRPODataset(tokenizer)


### 설정 및 값 준비: `dataloader`


In [ ]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


### 설정 및 값 준비: `data_iter`


In [ ]:
data_iter = cycle(dataloader)


### 설정 및 값 준비: `accuracies`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
accuracies = []


### 설정 및 값 준비: `current_accuracy`


In [ ]:
current_accuracy = 0.0


### 설정 및 값 준비: `pbar`


In [ ]:
pbar = tqdm(range(max_iters))


### 반복 실행


In [ ]:

for i in pbar:
    # 이 코드 단계의 동작을 확인하는 예시
    prompts, gts = next(data_iter)

    # 이 코드 단계의 동작을 확인하는 예시
    old_model.load_state_dict(model.state_dict())

    # 이 코드 단계의 동작을 확인하는 예시
    all_prompts, all_responses, all_advantages = generate_group(
        old_model, tokenizer, prompts, gts, group_size
    )

    # 이 코드 단계의 동작을 확인하는 예시
    ids, mask = dataset.get_batch(all_prompts, all_responses, device)
    all_advantages = all_advantages.to(device)

    # 이 코드 단계의 동작을 확인하는 예시
    for _ in range(n_update_per_generation):
        optimizer.zero_grad()
        loss = grpo_loss(model, old_model, ids, mask, all_advantages, epsilon)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 이 코드 단계의 동작을 확인하는 예시
        optimizer.step()

    # 이 코드 단계의 동작을 확인하는 예시
    if i % eval_interval == 0:
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for prompt, gt in dataset.data:
                response = generate(model, tokenizer, prompt, temperature=0)
                reward = calculate_reward(gt, response)
                correct += reward > 0
                total += 1
        model.train()
        current_accuracy = correct / total * 100
        accuracies.append(current_accuracy)

    pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_accuracy:.1f}%'})


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model.save(grpo_model_save_path)


### 실행 및 결과 확인


In [ ]:

plt.figure()


### 설정 및 값 준비: `steps`


In [ ]:
steps = list(range(0, len(accuracies) * eval_interval, eval_interval))


### 실행 및 결과 확인


In [ ]:
plt.plot(steps, accuracies)


### 실행 및 결과 확인


In [ ]:
plt.xlabel('Iteration')


### 실행 및 결과 확인


In [ ]:
plt.ylabel('Accuracy (%)')


### 실행 및 결과 확인


In [ ]:
plt.title('GRPO Training')


### 실행 및 결과 확인


In [ ]:
plt.grid(True)


### 실행 및 결과 확인


In [ ]:
plt.tight_layout()


### 실행 및 결과 확인


In [ ]:
plt.savefig("loss_grpo.png")


## T4 실행 메모

위 코드는 공식 구현의 모델 구조·알고리즘·기본 하이퍼파라미터를 보존합니다. 학습 시간이 긴 셀은 T4에서도 실행 자체는 가능할 수 있지만 전체 스텝 완주에는 시간이 많이 필요할 수 있습니다. 이 노트북은 빠른 실행을 위해 모델을 임의로 축소하거나 핵심 계산을 생략하지 않습니다.
